# 00 — Boîte à outils du terminal

Toutes les commandes du projet, prêtes à lancer : **plus rien à recopier**.

**Mode d'emploi.** Exécuter une cellule avec **Shift + Entrée**. Les cellules marquées
**★ SOUVENT** servent à chaque session. Les autres ne servent qu'une fois, ou en cas de
problème.

**Sécurité.** Rien n'est supprimé, déplacé ni poussé sans votre accord : chaque cellule
qui modifie quelque chose a un interrupteur (`SUPPRIMER`, `DEPLACER`, `POUSSER`…) à
`False`. Elle montre d'abord ce qu'elle *ferait*. On relit, puis on passe à `True` et on
relance la cellule.

**Ordre conseillé à chaque session :** 1 → 2 → 10, puis 8 → 9 avant un push, et 11 pour la présentation.

In [ ]:
# Reglages communs et outil pour lancer une commande — TOUJOURS executer en premier
import os
import shutil
import subprocess
import sys
from pathlib import Path

WORK = Path.home() / "work"
PROJET = WORK / "mesure-qualite-rp"
NOTEBOOKS = PROJET / "notebooks"
BELFORT = WORK / "pylaia-travail" / "modeles" / "pylaia-belfort"
ENV_SH = WORK / "env.sh"

# Le paquet Python du projet (dans src/) : c'est la que vit le code reutilisable
_paquets = sorted(p.name for p in (PROJET / "src").glob("*") if (p / "__init__.py").exists())
PKG = next((p for p in _paquets if (PROJET / "src" / p / "evaluation").is_dir()),
           _paquets[0] if _paquets else None)
SUIVI = PROJET / "src" / PKG / "evaluation" / "suivi.py" if PKG else None
CONVENTIONS = PROJET / "docs" / "CONVENTIONS.md"
PRESENTATION = PROJET / "docs" / "presentations" / "2026-10-14_point_etape.qmd"


def importer_suivi():
    """Importe suivi : src/<paquet>/evaluation/ (sa place), ou notebooks/ avant rangement."""
    import importlib
    if SUIVI and SUIVI.exists():
        sys.path.insert(0, str(PROJET / "src"))
        return importlib.import_module(f"{PKG}.evaluation.suivi")
    if (NOTEBOOKS / "suivi.py").exists():
        sys.path.insert(0, str(NOTEBOOKS))
        return importlib.import_module("suivi")
    raise ModuleNotFoundError("suivi.py introuvable (ni dans src/, ni dans notebooks/)")


def sh(commande, dossier=PROJET, delai=600):
    """Lance une commande bash et affiche tout ce qu'elle dit."""
    print(f"$ {commande}")
    try:
        r = subprocess.run(commande, shell=True, cwd=dossier if Path(dossier).exists() else None,
                           capture_output=True, text=True, timeout=delai,
                           executable="/bin/bash", env=os.environ.copy())
    except subprocess.TimeoutExpired:
        print(f"[arretee apres {delai} s]")
        return None
    sortie = (r.stdout + r.stderr).strip()
    if sortie:
        print(sortie)
    if r.returncode != 0:
        print(f"[code retour {r.returncode}]")
    return r


def taille(p):
    """Taille d'un fichier ou dossier, en Go."""
    p = Path(p)
    if p.is_file():
        return p.stat().st_size / 1e9
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file() and not f.is_symlink()) / 1e9


print("projet   :", PROJET, "(present)" if PROJET.exists() else "(ABSENT)")
print("paquet   :", PKG or "(aucun paquet trouve dans src/)")
print("python   :", sys.executable)

---
## 1. ★ SOUVENT — Au début de chaque session : les réglages d'environnement

Met le cache `uv`, l'environnement Python et le cache Hugging Face dans **`/tmp`** (grand
disque), et non dans `~/work` (petit disque). Sans ce réglage, une installation remplit
`~/work` et tout s'arrête (« No space left on device »).

Le fichier `~/work/env.sh` survit aux redémarrages du service. La cellule le crée s'il
manque, et applique les réglages à ce notebook. Dans un **terminal**, il faut en plus taper
une fois : `source ~/work/env.sh`

In [ ]:
CONTENU_ENV = """export UV_CACHE_DIR=/tmp/uv-cache
export UV_PROJECT_ENVIRONMENT=/tmp/mqrp-venv
export UV_LINK_MODE=copy
export HF_HOME=/tmp/hf-cache
export UV_TOOL_DIR=$HOME/work/uv-tools
export UV_TOOL_BIN_DIR=$HOME/work/bin
export PATH=$HOME/work/bin:$PATH
"""

if not ENV_SH.exists() or ENV_SH.read_text() != CONTENU_ENV:
    ENV_SH.write_text(CONTENU_ENV)
    print("env.sh ecrit :", ENV_SH)

bashrc = Path.home() / ".bashrc"
ligne = "source ~/work/env.sh"
if ligne not in (bashrc.read_text() if bashrc.exists() else ""):
    with open(bashrc, "a") as f:
        f.write(f"\n{ligne}\n")
    print("ajoute a ~/.bashrc (les nouveaux terminaux le chargeront)")

# Applique les reglages a ce notebook
for l in CONTENU_ENV.splitlines():
    cle, valeur = l.removeprefix("export ").split("=", 1)
    os.environ[cle] = os.path.expandvars(valeur)

for cle in ["UV_CACHE_DIR", "UV_PROJECT_ENVIRONMENT", "HF_HOME", "UV_TOOL_BIN_DIR"]:
    print(f"{cle:24} = {os.environ[cle]}")

---
## 2. ★ SOUVENT — Place sur les disques

`~/work` est le petit disque persistant. `/tmp` est grand, mais vidé à chaque redémarrage.
**En dessous de 2 Go libres sur `~/work`, faire le ménage (cellule 3).**

In [ ]:
for nom, chemin in [("~/work", WORK), ("/tmp", Path("/tmp"))]:
    u = shutil.disk_usage(chemin)
    alerte = "  <-- FAIRE LE MENAGE" if nom == "~/work" and u.free / 1e9 < 2 else ""
    print(f"{nom:8} libre {u.free/1e9:6.1f} Go sur {u.total/1e9:6.1f} Go{alerte}")

print("\nLes 15 elements les plus lourds de ~/work :")
sh("du -sh ~/work/* ~/work/.[!.]* 2>/dev/null | sort -h | tail -15", dossier=WORK, delai=180)

---
## 3. Ménage du disque `~/work`

Ne vise que ce qui est **sans risque** : les caches et les restes d'installation, qui se
recréent tout seuls, et les points de sauvegarde de la validation croisée (les mesures sont
dans MLflow).

**Laisser `SUPPRIMER = False`, lire la liste, puis passer à `True` et relancer.**

In [ ]:
SUPPRIMER = False

candidats = [
    WORK / ".cache" / "uv",          # ancien cache uv (desormais dans /tmp)
    PROJET / ".venv",                # environnement a moitie installe (desormais dans /tmp)
    *sorted(NOTEBOOKS.glob("trocr_b[0-9]*")),   # points de sauvegarde de la validation croisee
]
candidats = [c for c in candidats if c.exists()]

if not candidats:
    print("rien a supprimer")
total = 0
for c in candidats:
    t = taille(c)
    total += t
    print(f"  {t:6.2f} Go  {c}")
    if SUPPRIMER:
        shutil.rmtree(c, ignore_errors=True) if c.is_dir() else c.unlink()
if candidats:
    print(f"  {total:6.2f} Go au total —", "SUPPRIMES" if SUPPRIMER else
          "rien supprime (passer SUPPRIMER = True pour le faire)")

print("\nA examiner a la main (pas supprimes automatiquement) :")
for p in [WORK / ".cache" / "huggingface", WORK / "pylaia-travail" / "cv-bloc1",
          WORK / "pylaia-travail" / "test-zero", WORK / "pylaia-travail" / "test-surapprentissage",
          NOTEBOOKS / "trocr_ajuste", NOTEBOOKS / "trocr_bloc1"]:
    if p.exists():
        print(f"  {taille(p):6.2f} Go  {p}")

---
## 4. Le dossier du modèle Belfort doit rester d'origine

Tout fichier ajouté dans `pylaia-belfort` (par exemple `weights_reset.ckpt`) risque d'être
pris pour les vrais poids. Cette cellule les repère et, si `DEPLACER = True`, les range dans
`pylaia-travail/archives/`. Rien n'est supprimé.

In [ ]:
DEPLACER = False

ORIGINE = {"language_model.arpa.gz", "lexicon.txt", "model", "README.md",
           "syms.txt", "tokens.txt", "weights.ckpt"}

if not BELFORT.exists():
    print("dossier Belfort absent :", BELFORT)
else:
    presents = {p.name for p in BELFORT.iterdir()}
    intrus = sorted(presents - ORIGINE)
    manquants = sorted(ORIGINE - presents)
    print("fichiers d'origine manquants :", manquants or "aucun")
    print("fichiers ajoutes             :", intrus or "aucun")
    if intrus and DEPLACER:
        archives = BELFORT.parent.parent / "archives"
        archives.mkdir(exist_ok=True)
        for nom in intrus:
            shutil.move(str(BELFORT / nom), str(archives / nom))
            print(f"  deplace -> {archives / nom}")
    elif intrus:
        print("(passer DEPLACER = True pour les ranger dans archives/)")

---
## 5. Une seule fois — nbstripout

Retire automatiquement les sorties des notebooks **dans ce qui part sur Git**. Votre copie
locale garde ses sorties. Installé comme outil séparé : ne déclenche **pas** l'installation
de torch.

Nécessite la cellule 1 (pour que l'outil s'installe dans `~/work/bin`, qui survit aux
redémarrages).

In [ ]:
deja = sh("git config --get filter.nbstripout.clean")
if deja is not None and deja.returncode == 0 and deja.stdout.strip():
    print("\nnbstripout est deja branche sur Git : rien a faire")
else:
    sh("uv tool install nbstripout", delai=300)
    exe = Path(os.environ.get("UV_TOOL_BIN_DIR", WORK / "bin")) / "nbstripout"
    if exe.exists():
        sh(f"{exe} --install")
        sh("git config --get filter.nbstripout.clean")
    else:
        print("nbstripout introuvable apres installation :", exe)

---
## 6. Une seule fois — Ce que Git doit ignorer

Les résultats (CSV), les identifiants de bulletins, les pages HTML et les poids de modèles
ne vont **jamais** sur Git : ils vont dans MLflow, ou se recalculent. La cellule ajoute ce
qui manque au `.gitignore`, sans rien dupliquer.

In [ ]:
A_IGNORER = [
    "notebooks/*.csv",
    "notebooks/*.html",
    "notebooks/*_files/",
    "docs/presentations/*.html",
    "docs/presentations/*_files/",
    "notebooks/bulletins_test_*.txt",
    "notebooks/trocr_*/",
    "notebooks/pylaia/",
    "*.ckpt",
    "*.safetensors",
    ".venv/",
]
gi = mesure-qualite-rp/ ".gitignore"
existant = gi.read_text().splitlines() if gi.exists() else []
ajouts = [l for l in A_IGNORER if l not in existant]
if ajouts:
    with open(gi, "a") as f:
        f.write(("\n" if existant and existant[-1] else "") + "\n".join(ajouts) + "\n")
    print("ajoute au .gitignore :", *ajouts, sep="\n  ")
else:
    print(".gitignore deja complet")

---
## 6 bis. Une seule fois — Ranger les fichiers à leur place, sur une branche dédiée

| Fichier | Va dans | Pourquoi |
|---|---|---|
| `suivi.py` | `src/<paquet>/evaluation/` | code de mesure, réutilisé par les notebooks et la présentation |
| `CONVENTIONS.md` | `docs/` | les règles du projet, pour tous |
| `presentation_14oct.qmd` | `docs/presentations/2026-10-14_point_etape.qmd` | un livrable, daté |
| notebooks `00`, `05` | `notebooks/` | l'analyse et les outils |

Le travail part sur une **branche dédiée**, pour être relu (merge request) avant d'arriver
sur la branche principale. Laisser `RANGER = False`, lire, puis `True` et relancer.

In [ ]:
RANGER = True
BRANCHE = "feat/suivi-mlflow"

if not PKG:
    print("aucun paquet dans src/ : demander a Cedric ou mettre le code partage")
else:
    deplacements = [
        (NOTEBOOKS / "suivi.py", SUIVI),
        (NOTEBOOKS / "presentation_14oct.qmd", PRESENTATION),
        (NOTEBOOKS / "CONVENTIONS.md", CONVENTIONS),
    ]
    actuelle = sh("git branch --show-current")
    actuelle = actuelle.stdout.strip() if actuelle is not None else ""
    print()
    if actuelle != BRANCHE:
        print(f"branche : {actuelle} -> {BRANCHE}")
    for src_, dst in deplacements:
        if src_.exists():
            print(f"deplacer : {src_.relative_to(PROJET)} -> {dst.relative_to(PROJET)}")
        elif dst.exists():
            print(f"deja en place : {dst.relative_to(PROJET)}")
    if RANGER:
        if actuelle != BRANCHE:
            existe = sh(f"git rev-parse --verify --quiet {BRANCHE}")
            sh(f"git switch {BRANCHE}" if existe is not None and existe.returncode == 0
               else f"git switch -c {BRANCHE}")
        for src_, dst in deplacements:
            if src_.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                if dst.suffix == ".py" and not (dst.parent / "__init__.py").exists():
                    (dst.parent / "__init__.py").touch()
                suivi_git = sh(f"git ls-files --error-unmatch \"{src_}\"")
                if suivi_git is not None and suivi_git.returncode == 0:
                    sh(f"git mv \"{src_}\" \"{dst}\"")
                else:
                    shutil.move(str(src_), str(dst))
        print("\nrange. Suite : cellule 8 (verifier), puis 9 (commit et push).")
    else:
        print("\n(passer RANGER = True pour le faire)")

---
## 7. Environnement Python et GPU

Pour vérifier quel Python et quelles versions tournent, et si le GPU est visible.

In [ ]:
print("python :", sys.executable)
for module in ["torch", "transformers", "PIL", "pandas", "mlflow"]:
    try:
        m = __import__(module)
        print(f"{module:13} {getattr(m, '__version__', '?')}")
    except Exception:
        print(f"{module:13} absent")
try:
    import torch
    print("GPU visible  :", torch.cuda.is_available())
except Exception:
    pass
sh("nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv", delai=30)

---
## 8. ★ SOUVENT — Avant un push : vérifier ce qui partirait

Montre la branche, les fichiers modifiés, et **signale tout fichier interdit** (CSV, HTML,
identifiants, poids). S'il y a une alerte : ne pas pousser.

In [ ]:
INTERDITS = (".csv", ".html", ".ckpt", ".safetensors", ".parquet", ".xlsx")


def fichiers_interdits(liste):
    return [f for f in liste if f.endswith(INTERDITS) or "bulletins_test_" in f]


sh("git branch --show-current")
r = sh("git status --short")
modifies = [l[3:].strip() for l in (r.stdout.splitlines() if r is not None else [])]
alerte = fichiers_interdits(modifies)
print()
if alerte:
    print("ALERTE — fichiers qui ne doivent pas aller sur Git :", *alerte, sep="\n  ")
    print("-> lancer la cellule 6, puis relancer celle-ci")
else:
    print("aucun fichier interdit en vue")

---
## 9. ★ SOUVENT — Commit et push

**Un commit = un changement**, avec un message qui dit *quoi* et *pourquoi*. Mettre dans
`LOTS` un lot par changement : chaque lot devient un commit, puis tout part en un push.

1. Lancer une première fois avec `POUSSER = False` : la cellule vérifie et montre les lots.
2. Si tout est propre, passer `POUSSER = True` et relancer.

La cellule **refuse** de committer un fichier interdit.

**Rappel MLflow** : committer **avant** de lancer un essai qu'on compte retenir. Ainsi,
l'essai enregistre une version du code qui existe vraiment sur Git.

In [ ]:
LOTS = [
    ([".gitignore"],
     "chore: ignorer resultats, identifiants de bulletins et poids de modeles"),
    (["docs/CONVENTIONS.md"],
     "docs: conventions du projet (rangement, MLflow, Quarto, Git, statistiques)"),
    ([f"src/{PKG}/evaluation/suivi.py"],
     "feat(suivi): une experience MLflow, etiquettes par etude, un essai retenu par etude"),
    (["notebooks/05_cv_et_complementarite.ipynb"],
     "feat(nb05): bascule sur ILLISIBLE et enregistrement des resultats retenus"),
    (["docs/presentations/2026-10-14_point_etape.qmd"],
     "docs: presentation du 14 octobre, chiffres lus dans MLflow"),
    (["notebooks/00_outils_terminal.ipynb"],
     "chore: boite a outils du terminal"),
]
POUSSER = True

a_committer = []
for fichiers, message in LOTS:
    existants = [f for f in fichiers if (PROJET / f).exists()]
    for f in fichiers:
        if f not in existants:
            print("absent (ignore) :", f)
    if fichiers_interdits(existants):
        print("REFUS — fichiers interdits :", fichiers_interdits(existants))
        a_committer = []
        break
    if existants:
        a_committer.append((existants, message))

if a_committer and not POUSSER:
    print("\nLots prets :")
    for existants, message in a_committer:
        print(f"  [{message}]", *existants, sep="\n      ")
    print("\n-> relire, puis passer POUSSER = True et relancer")
elif a_committer:
    for existants, message in a_committer:
        sh("git add " + " ".join(f'"{f}"' for f in existants))
        prets = sh("git diff --cached --name-only")
        prets = prets.stdout.split() if prets is not None else []
        if fichiers_interdits(prets):
            print("REFUS — fichiers interdits prepares :", fichiers_interdits(prets))
            sh("git reset -q")
            break
        if not prets:
            print("  (rien de nouveau pour ce lot)")
            continue
        sh(f'git commit -q -m "{message.replace(chr(34), chr(39))}"')
    sh("git log --oneline -8")
    sh("git push -u origin HEAD", delai=120)

---
## 10. ★ SOUVENT — Où en sont les essais (MLflow)

Un essai par ligne, le plus récent en haut, avec son statut (`retenu`, `exploratoire`,
`remplace`, `abandonne`). Seuls les essais lancés avec `suivi.py` apparaissent.

In [ ]:
try:
    suivi = importer_suivi()
    vue = suivi.tableau()
    display(vue) if len(vue) else None
except ModuleNotFoundError as e:
    print("module manquant :", e.name,
          "(suivi.py introuvable, ou mlflow absent)")
except Exception as e:
    print("MLflow ne repond pas :", type(e).__name__, e)

---
## 11. ★ SOUVENT — Fabriquer la présentation

Relit les essais **retenus** dans MLflow et produit le `.html` à côté du `.qmd`, dans `docs/presentations/`. Ensuite,
dans l'explorateur : clic droit sur le `.html` → **Download**, puis l'ouvrir dans Firefox.
(Ne pas l'ouvrir dans JupyterLab : les diapositives ne s'afficheraient pas.)

In [ ]:
if not shutil.which("quarto"):
    print("quarto n'est pas installe sur ce service : demander a Cedric comment il l'utilise")
elif not PRESENTATION.exists():
    print("fichier absent :", PRESENTATION, "(lancer d'abord la cellule de rangement)")
else:
    r = sh(f"quarto render {PRESENTATION.name}", dossier=PRESENTATION.parent, delai=600)
    html = PRESENTATION.with_suffix(".html")
    if r is not None and r.returncode == 0 and html.exists():
        print(f"\nOK : {html}  ({html.stat().st_size/1e6:.1f} Mo)")

---
## 12. Rarement — Réinstaller l'environnement du projet

**Lourd** : torch et ses bibliothèques font plusieurs gigaoctets. La cellule vérifie d'abord
que l'installation ira dans `/tmp` (cellule 1) et qu'il y a assez de place, sinon elle
refuse. À ne lancer que si un import échoue après un redémarrage.

In [ ]:
INSTALLER = False
PLACE_MINIMUM_GO = 15

cible = os.environ.get("UV_PROJECT_ENVIRONMENT", "")
libre_tmp = shutil.disk_usage("/tmp").free / 1e9
print(f"environnement cible : {cible or '(non regle)'}")
print(f"libre sur /tmp      : {libre_tmp:.1f} Go")

if not cible.startswith("/tmp"):
    print("REFUS — lancer d'abord la cellule 1 (sinon tout irait dans ~/work)")
elif libre_tmp < PLACE_MINIMUM_GO:
    print(f"REFUS — moins de {PLACE_MINIMUM_GO} Go libres sur /tmp")
elif not INSTALLER:
    print("pret : passer INSTALLER = True pour lancer uv sync (plusieurs minutes)")
else:
    sh("uv sync", delai=1800)